In [3]:
import sys
import os
import pathlib
import glob
import pandas as pd

In [6]:
# Obter o caminho absoluto do notebook atual
notebook_path = pathlib.Path().absolute()
print(f"Localização atual do notebook: {notebook_path}")

# Subir um nível para chegar à pasta Modelo1 (projeto root)
root_folder = notebook_path.parent
print(f"Pasta raiz do projeto (root_folder): {root_folder}")

# Definir caminhos baseado na estrutura do projeto
SRC_DIR = root_folder / "src/data_processing"
DATA_DIR = root_folder / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
COMPLETE_SERIES_DIR = DATA_DIR / "complete_series"
AUXILIARY_COMPLETE_SERIES_DIR = DATA_DIR / "auxiliary_complete_series"

# Adicionar src ao path para importar os módulos personalizados
sys.path.append(str(SRC_DIR))

# Verificar se os diretórios existem
print("\nVerificando estrutura de diretórios:")
print(f"✓ Pasta src: {SRC_DIR} {'(existe)' if SRC_DIR.exists() else '(não existe)'}")
print(f"✓ Pasta data: {DATA_DIR} {'(existe)' if DATA_DIR.exists() else '(não existe)'}")
print(f"✓ Pasta raw: {RAW_DATA_DIR} {'(existe)' if RAW_DATA_DIR.exists() else '(não existe)'}")

# Criar diretórios se não existirem
for dir_path in [COMPLETE_SERIES_DIR, AUXILIARY_COMPLETE_SERIES_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"✓ Pasta criada/verificada: {dir_path}")


Localização atual do notebook: c:\Users\emily\Documents\TCC_projeto_teste\Modelo1\notebooks
Pasta raiz do projeto (root_folder): c:\Users\emily\Documents\TCC_projeto_teste\Modelo1

Verificando estrutura de diretórios:
✓ Pasta src: c:\Users\emily\Documents\TCC_projeto_teste\Modelo1\src\data_processing (existe)
✓ Pasta data: c:\Users\emily\Documents\TCC_projeto_teste\Modelo1\data (existe)
✓ Pasta raw: c:\Users\emily\Documents\TCC_projeto_teste\Modelo1\data\raw (existe)
✓ Pasta criada/verificada: c:\Users\emily\Documents\TCC_projeto_teste\Modelo1\data\complete_series
✓ Pasta criada/verificada: c:\Users\emily\Documents\TCC_projeto_teste\Modelo1\data\auxiliary_complete_series


In [7]:
#  Importar módulos personalizados
try:
    from complete_series import DataPreprocessor
    from interpolate_series import batch_interpolate_and_overwrite
    print("\n✓ Módulos importados com sucesso!")
except ImportError as e:
    print(f"\n✗ Erro ao importar módulos: {e}")
    print("Verifique se os arquivos estão em:", SRC_DIR)


✓ Módulos importados com sucesso!


In [ ]:
#  Completar séries temporais
# Listas de estações (AJUSTE ESTES VALORES CONFORME SUA NECESSIDADE)
main_stations = [10100000, 10200000, 10300000]  # Exemplo - séries principais
auxiliary_stations = [10500000, 11400000, 11500000]  # Exemplo - séries auxiliares

# Mostrar informações
print(f"Diretório de dados brutos: {RAW_DATA_DIR}")
print(f"Número de estações principais: {len(main_stations)}")
print(f"Número de estações auxiliares: {len(auxiliary_stations)}")
print(f"Total de estações a processar: {len(main_stations) + len(auxiliary_stations)}")

# Instanciar preprocessador
preprocessor = DataPreprocessor(csv_path=str(RAW_DATA_DIR))

# Processar e salvar estações nas pastas apropriadas
try:
    main_data, auxiliary_data = preprocessor.process_and_save_stations(
        main_stations=main_stations,
        auxiliary_stations=auxiliary_stations,
        main_output_path=str(COMPLETE_SERIES_DIR),
        auxiliary_output_path=str(AUXILIARY_COMPLETE_SERIES_DIR),
        start_date='1995-01-01',
        end_date='2012-04-30'
    )
    
    print(f"\n✓ Processamento concluído:")
    print(f"  - Séries principais: {len(main_data)} estações em '{COMPLETE_SERIES_DIR}'")
    print(f"  - Séries auxiliares: {len(auxiliary_data)} estações em '{AUXILIARY_COMPLETE_SERIES_DIR}'")
    
except Exception as e:
    print(f"\n✗ Erro durante o processamento: {e}")

In [ ]:
# Interpolar séries (apenas para estações específicas)
# Configuração das estações que precisam de interpolação
# (Pode ser uma lista vazia se nenhuma estação precisar)
stations_to_interpolate = [
    {
        'target_id': 10100000,
        'variables': ['precipitation_chirps', 'potential_evapotransp_gleam'],
        'neighbor_ids': [10500000, 11400000],
        'distances_km': [107.319, 143.381],
        'power': 1.0
    }
    # Adicionar mais configurações se necessário
]

# Verificar se os arquivos de entrada existem
print("Verificando arquivos necessários para interpolação...")
for config in stations_to_interpolate:
    target_file = COMPLETE_SERIES_DIR / f"{config['target_id']}_complete_date.csv"
    print(f"\nEstação alvo {config['target_id']}:")
    print(f"  Arquivo: {target_file} {'✓' if target_file.exists() else '✗ (não encontrado)'}")
    
    for neighbor_id in config['neighbor_ids']:
        neighbor_file = AUXILIARY_COMPLETE_SERIES_DIR / f"{neighbor_id}_complete_date.csv"
        print(f"  Vizinho {neighbor_id}: {neighbor_file} {'✓' if neighbor_file.exists() else '✗'}")
    print("-" * 40)

if stations_to_interpolate:
    try:
        results = batch_interpolate_and_overwrite(
            station_configs=stations_to_interpolate,
            target_data_dir=str(COMPLETE_SERIES_DIR),
            neighbor_data_dir=str(AUXILIARY_COMPLETE_SERIES_DIR),
            verbose=True
        )
        
        print(f"\n✓ Interpolação concluída para {len(results)} estações")
        print(f"  Arquivos atualizados na pasta: {COMPLETE_SERIES_DIR}")
        
    except Exception as e:
        print(f"\n✗ Erro durante a interpolação: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Nenhuma estação configurada para interpolação.")

In [ ]:
# Verificação dos resultados
# Verificar se os arquivos foram criados/corretamente
# Listar arquivos gerados
print(f"Conteúdo de {COMPLETE_SERIES_DIR}:")
complete_files = list(COMPLETE_SERIES_DIR.glob("*.csv"))
for i, f in enumerate(sorted(complete_files), 1):
    try:
        df = pd.read_csv(f, nrows=0)  # Ler apenas cabeçalho
        cols = list(df.columns)
        print(f"{i:2d}. {f.name} ({len(cols)} colunas): {', '.join(cols[:3])}...")
    except:
        print(f"{i:2d}. {f.name} (erro ao ler)")

print(f"\nConteúdo de {AUXILIARY_COMPLETE_SERIES_DIR}:")
auxiliary_files = list(AUXILIARY_COMPLETE_SERIES_DIR.glob("*.csv"))
for i, f in enumerate(sorted(auxiliary_files), 1):
    try:
        df = pd.read_csv(f, nrows=0)
        cols = list(df.columns)
        print(f"{i:2d}. {f.name} ({len(cols)} colunas): {', '.join(cols[:3])}...")
    except:
        print(f"{i:2d}. {f.name} (erro ao ler)")

In [ ]:
# Resumo estatístico (opcional)
# Função para mostrar estatísticas básicas de um arquivo
def show_file_stats(file_path, max_rows=5):
    try:
        df = pd.read_csv(file_path)
        print(f"\n📊 Estatísticas para: {file_path.name}")
        print(f"   Período: {df['date'].min()} a {df['date'].max()}")
        print(f"   Total de dias: {len(df)}")
        print(f"   Colunas: {list(df.columns)}")
        print(f"   Valores faltantes por coluna:")
        for col in df.columns:
            if col != 'date' and col != 'station_id':
                missing = df[col].isna().sum()
                if missing > 0:
                    percent = (missing / len(df)) * 100
                    print(f"     - {col}: {missing} faltantes ({percent:.1f}%)")
        print("-" * 40)
    except Exception as e:
        print(f"   Erro ao ler {file_path.name}: {e}")

# Mostrar estatísticas para algumas estações
print("\nEstatísticas para estações principais (amostra):")
for i, file_path in enumerate(complete_files[:3]):  # Mostrar apenas as 3 primeiras
    show_file_stats(file_path)

print("\nEstatísticas para estações auxiliares (amostra):")
for i, file_path in enumerate(auxiliary_files[:3]):  # Mostrar apenas as 3 primeiras
    show_file_stats(file_path)